# 01 — TransformerLens Basics

**Objetivo (días 3–4):** cargar GPT-2, ejecutar una inferencia y extraer activaciones de una capa intermedia.

**Entregable:** mostrar las activaciones de la capa 10 como tensor con su shape.

## Diccionario de conceptos

---

**Transformer**  
Arquitectura de red neuronal basada en mecanismos de atención. Procesa secuencias de tokens en paralelo. GPT-2 es un transformer decoder-only: genera texto token a token, cada nuevo token condicionado en todos los anteriores.

---

**Token**  
Unidad mínima de texto que procesa el modelo. No equivale a una palabra: puede ser una sílaba, una palabra completa o un signo de puntuación. GPT-2 tiene un vocabulario de 50.257 tokens.

---

**BPE (Byte-Pair Encoding)**  
Algoritmo de tokenización que divide el texto en subpalabras frecuentes. "Eiffel" puede ser uno o varios tokens según cuánto aparece en el corpus de preentrenamiento. Es el algoritmo que usa GPT-2.

---

**Embedding**  
Representación de un token como vector de números reales. GPT-2 mapea cada token a un vector de 768 dimensiones. Los embeddings posicionales añaden información sobre la posición del token en la secuencia.

---

**d_model**  
Dimensión del espacio de representación interno del modelo. En GPT-2 small, `d_model = 768`. Todos los tensores que fluyen por el modelo tienen esta dimensión en su eje de representación. Es el tamaño del residual stream.

---

**Residual stream**  
El tensor principal que fluye a través de todas las capas del transformer. En lugar de que cada capa reemplace la representación anterior, cada componente (attention, MLP) *suma* su contribución al mismo tensor:

```
x₀ = embedding(tokens)         # (batch, seq_len, 768)
x₁ = x₀ + attn_0(x₀)          # attention capa 0 suma su delta
x₂ = x₁ + mlp_0(x₁)           # MLP capa 0 suma su delta
x₃ = x₂ + attn_1(x₂)          # ...
...
logits = unembed(x_final)       # proyección al vocabulario
```

El residual stream acumula información a lo largo de todas las capas. Esto es clave para el steering: si sumamos un vector al residual stream en la capa 10, todas las capas posteriores lo procesarán como si fuera información real.

---

**Attention head**  
Subcomponente de una capa de atención. Cada capa de GPT-2 tiene 12 attention heads en paralelo. Cada head aprende a relacionar tokens entre sí de una forma diferente (sintaxis, correferencia, semántica...). Sus outputs se concatenan y se proyectan de vuelta al residual stream.

---

**MLP (Multi-Layer Perceptron)**  
El segundo subcomponente de cada capa transformer, después de la atención. Opera token a token (sin cruzar información entre posiciones). Funciona como memoria factual: almacena asociaciones entre patrones y conocimiento. Cada capa transformer tiene una atención y un MLP.

---

**Forward pass**  
Una pasada completa de los datos a través del modelo, de entrada a salida. Produce los logits. En generación autoregresiva se hace un forward pass por cada token generado.

---

**Logits**  
Salida cruda del modelo antes del softmax. Un vector de 50.257 valores (uno por token del vocabulario). Aplicando softmax se obtiene la distribución de probabilidad sobre el siguiente token. El token con mayor logit es la predicción más probable.

---

**Activaciones**  
Los valores intermedios que produce el modelo durante el forward pass: embeddings, outputs de attention, outputs de MLP, residual stream en cada capa... Las activaciones son el objeto de estudio de la interpretabilidad mecanicista — analizar qué representan permite entender qué hace el modelo internamente.

---

**HookedTransformer**  
La clase principal de TransformerLens. Reimplementa modelos conocidos (GPT-2, GPT-J, Llama...) con un sistema de hooks integrado. `from_pretrained()` descarga los pesos desde Hugging Face y los reformatea para que el sistema de hooks funcione.

---

**`run_with_cache()`**  
Método de TransformerLens que ejecuta el modelo y guarda automáticamente *todas* las activaciones intermedias en un diccionario (`cache`). Equivale a registrar un hook de lectura en cada punto del grafo computacional. Úsalo para inspeccionar; en producción usa `run_with_hooks()` para no guardar todo en memoria.

---

**Hook**  
Función que se registra en un punto específico del grafo computacional y se ejecuta automáticamente cada vez que el modelo pasa por ese punto. Puede leer las activaciones (para inspeccionarlas) o modificarlas (para intervenir). Es el mecanismo que hace posible el activation steering sin tocar los pesos del modelo.

---

**Cache**  
Diccionario que devuelve `run_with_cache()` con todas las activaciones guardadas. Las claves siguen una nomenclatura consistente: `"resid_post", 10` es el residual stream después de la capa 10; `"blocks.5.attn.hook_z"` son los outputs de las attention heads de la capa 5.

---

**Interpretabilidad mecanicista**  
Campo de investigación que busca entender qué computan internamente los modelos de lenguaje — qué representan las activaciones, qué hace cada cabeza de atención, cómo fluye la información. TransformerLens es una herramienta diseñada específicamente para este campo. El activation steering es una forma de interpretabilidad *intervencionista*: en lugar de solo observar, modifica y mide el efecto.

## 1. Instalación

**TransformerLens** es una librería de interpretabilidad para modelos transformer desarrollada por Neel Nanda. La diferencia clave con Hugging Face: TransformerLens expone el interior del modelo con un API diseñado para intervenir en él.

Lo que nos importa de TransformerLens:
- **Residual stream**: el tensor principal que fluye a través de las capas y acumula información
- **`run_with_cache()`**: ejecuta el modelo y guarda automáticamente todas las activaciones intermedias
- **Hooks**: funciones que se ejecutan en puntos específicos del grafo computacional — permiten leer *o modificar* activaciones en tiempo real

Esto es exactamente lo que necesitamos para activation steering: interceptar el flujo de información y modificarlo sin tocar los pesos del modelo.

In [1]:
!pip install transformer_lens -q

## 2. Cargar GPT-2

**Arquitectura de GPT-2 (small):**
- 12 capas transformer
- `d_model = 768` — dimensión del residual stream (el espacio donde "viven" los conceptos)
- 12 attention heads por capa, `d_head = 64`
- ~117M parámetros en total

`HookedTransformer.from_pretrained("gpt2")` descarga los pesos desde Hugging Face y los reformatea internamente para que el sistema de hooks funcione. El `model.eval()` desactiva dropout — necesario para que las activaciones sean deterministas entre ejecuciones.

In [2]:
import transformer_lens, torch
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2")
model.eval()
print(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): Ho

## 3. Inferencia básica

**Tokenización:** GPT-2 usa BPE (Byte-Pair Encoding). Las palabras se dividen en subpalabras frecuentes. "Eiffel" puede ser un token o varios, según cuánto aparece en el corpus de preentrenamiento. `to_str_tokens()` es útil para ver exactamente cómo el modelo "ve" el texto.

El tensor de tokens tiene shape `[batch, seq_len]` — el `1` es el batch size.

Los **logits** de salida tienen shape `[batch, seq_len, vocab_size]` = `[1, n_tokens, 50257]`. Para cada posición, el modelo predice una distribución sobre los ~50k tokens del vocabulario. El softmax de los logits de la última posición da la probabilidad del siguiente token.

In [3]:
prompt = "The doctor told the patient that he had cancer. The patient felt"
tokens = model.to_tokens(prompt)

print("Tokens:", tokens)
print("Token strings:", model.to_str_tokens(prompt))

logits = model(tokens)
print("Logits shape:", logits.shape)

Tokens: tensor([[50256,   464,  6253,  1297,   262,  5827,   326,   339,   550,  4890,
            13,   383,  5827,  2936]])
Token strings: ['<|endoftext|>', 'The', ' doctor', ' told', ' the', ' patient', ' that', ' he', ' had', ' cancer', '.', ' The', ' patient', ' felt']
Logits shape: torch.Size([1, 14, 50257])


## 4. Extraer activaciones — capa 10

**El residual stream** es el mecanismo central de los transformers modernos. En lugar de transformaciones secuenciales donde la salida de una capa reemplaza a la entrada, cada componente *suma* su contribución al mismo tensor:

```
x₀ = embedding(tokens)                 # shape: (batch, seq_len, 768)
x₁ = x₀ + attn_0(x₀)                  # capa 0 attention añade su delta
x₂ = x₁ + mlp_0(x₁)                   # capa 0 MLP añade su delta
x₃ = x₂ + attn_1(x₂)                  # capa 1 attention...
...
logits = unembed(x₂₄)                  # proyección final al vocabulario
```

`cache["resid_post", 10]` devuelve el residual stream *después* de que la capa 10 (attention + MLP) haya añadido su contribución. Shape: `(batch, seq_len, d_model)` = `(1, n_tokens, 768)`.

**Por qué esto importa para steering:** si una dirección en este espacio de 768 dimensiones codifica "escepticismo", podemos *añadir* esa dirección durante la inferencia y las capas posteriores lo interpretarán como contexto real — exactamente como si hubiera llegado naturalmente del procesamiento anterior.

In [4]:
logits, cache = model.run_with_cache(tokens)

# activaciones del residual stream tras la capa 10

activations_layer10 = cache["resid_post", 10]
print("Shape:", activations_layer10.shape)  # (1, n_tokens, 768)
print(activations_layer10)


Shape: torch.Size([1, 14, 768])
tensor([[[-5.2428, -4.7903, -5.1806,  ..., -3.4617, -3.4614, -5.0972],
         [-2.6659, -0.0799,  2.5872,  ..., -0.4059,  0.4512, -2.7703],
         [-0.6986,  0.2131, -5.9306,  ...,  7.8718, -0.2863, -4.3675],
         ...,
         [-2.0174, -4.8692,  1.0087,  ..., -0.2556,  0.2336, -1.6701],
         [-0.6132, -0.1947,  0.4350,  ...,  7.2819,  6.9770, -0.9203],
         [-9.9730, -1.8452, -1.2797,  ...,  2.9373, -0.0445, -4.0072]]])


## 5. Explorar la estructura de hooks

**Hooks en TransformerLens:** cada punto del grafo computacional tiene un nombre consistente. Los hooks permiten registrar funciones que se ejecutan en ese punto para leer o modificar activaciones.

Nomenclatura de las claves más importantes:
- `blocks.N.hook_resid_pre` — residual stream *antes* de la capa N
- `blocks.N.hook_resid_post` — residual stream *después* de la capa N (attention + MLP)
- `blocks.N.attn.hook_z` — output de las attention heads de la capa N
- `blocks.N.mlp.hook_post` — output del MLP de la capa N
- `hook_embed` — embeddings de tokens
- `hook_pos_embed` — embeddings posicionales

`run_with_cache()` activa todos los hooks en modo lectura y guarda los resultados. En el notebook 02 usaremos `model.generate()` con `hooks=` para *escribir* — inyectar un vector en el residual stream durante la inferencia.

In [5]:
# Ver todas las claves disponibles en la cache
print("Claves disponibles en cache:")
for key in list(cache.keys())[:20]:
    print(" ", key)

Claves disponibles en cache:
  hook_embed
  hook_pos_embed
  blocks.0.hook_resid_pre
  blocks.0.ln1.hook_scale
  blocks.0.ln1.hook_normalized
  blocks.0.attn.hook_q
  blocks.0.attn.hook_k
  blocks.0.attn.hook_v
  blocks.0.attn.hook_attn_scores
  blocks.0.attn.hook_pattern
  blocks.0.attn.hook_z
  blocks.0.hook_attn_out
  blocks.0.hook_resid_mid
  blocks.0.ln2.hook_scale
  blocks.0.ln2.hook_normalized
  blocks.0.mlp.hook_pre
  blocks.0.mlp.hook_post
  blocks.0.hook_mlp_out
  blocks.0.hook_resid_post
  blocks.1.hook_resid_pre
